# Fourierreihen mit Python [MUSTERLÖSUNG]

<div style = "background-color: whitesmoke; padding: 15px; border-radius: 8px; border: 2px solid black">
    
**Ausgabedatum:** 22.06.2023 <br>
**Abgabedautm:** 29.06.2023 <br>
Für Rückfragen zum Übungsbetrieb wenden Sie sich an: mustermann@uni-muster.de

In diesem Notebook werden Sie verschiedene Aspekte von Fourierreihen mit Python untersuchen. Das Ziel ist es zunächst eine Funktion zu definieren, welche die Fourierreihe einer mathematischen Funktion $f$ mithilfe von SciPy numerisch berechnet. Anschließend sollen die Fourierkoeffizienten eines Datensatzes bestimmt werden um die Parameter der zugrundeliegenden Schwingungen zu rekonstruieren. 

**Python Grundlagen:** Einfache Schleifen, Plots. <br>
**Math. Grundlagen:** Fourierreihen

Aus der Vorlesung sind Ihnen bereits **Fourierreihen** bekannt. Jede $2\pi$-periodische und auf $[0, 2\pi]$ integrierbare Funktion $f(t)$ lässt sich in eine Funktionenreihe aus Sinus- und Kosinusfunktionen entwickeln. Es gilt

\begin{equation*} 
f(t) = \frac{a_0}{2} + \sum_{k = 1}^{\infty} \left( a_k \cos (kt) + b_k \sin (kt) \right) \ .
\tag{1}\end{equation*}

Die **Fourierkoeffizienten** $a_k$ und $b_k$ sind gegeben durch

\begin{equation*}
a_k = \frac{1}{\pi} \int_{0}^{2\pi} \mathrm{d}\tau \ f(\tau)\cos (k\tau) \ , \quad b_k = \frac{1}{\pi} \int_{0}^{2\pi} \mathrm{d}\tau \ f(\tau)\sin (k\tau) \ .
\tag{2}\end{equation*}

> **Am Rande:** Das Konzept der Fourierreihen lässt sich auch auf Funktionen mit beliebiger Periodizität $T$ verallgemeinern. Dazu verwendet man die Substitution $\tilde{\tau} = 2\pi \tau/T$ und passt die Integrationsgrenzen und die Normierung entsprechend an. In diesem Notebook werden wir allerdings nur von $2\pi$-periodischen Funktionen ausgehen.

Wir wollen nun eine Pythonfunktion schreiben, welche uns die Fourierreihe für eine beliebige Funktion $f$ bis zur Ordnung $N$ numerisch berechnet. Diese wird den Namen ```MyFourier(func, t, N)``` tragen.

### I. Numerische Integration mit SciPy

Um die notwendigen Integrationen durchzuführen werden wir die Funktion ```quad(f, a, b)``` aus der Bibliothek *SciPy* verwenden. Hierbei bezeichnet $f$ den Integranten (Pythonfunktion) und $a$, $b$ die untere, bzw. obere Integrationsgrenze. Ein Beispiel:

$$ I = \int_{-1}^{1} \mathrm{d}x \ (x^2 + 1) = \left[\frac{1}{3}x^3 + x \right]_{-1}^{1} = \frac{1}{3} + 1 + \frac{1}{3} + 1 = \frac{8}{3} =  2.666...$$

Mit SciPy lässt sich das Integral wie folgt berechnen:

In [ ]:
from scipy.integrate import quad

def f(x):
    return x**2 + 1

I = quad(f, -1, 1)

print(I)

Wie man sieht, gibt die Funktion ```quad()``` zwei Werte zurück: den numerischen Wert des Integrals und den geschätzten Fehler auf den numerischen Wert. In unserem Fall ist dieser jedoch vernachlässigbar. Im nächsten Beispiel betrachten wir eine Funktion, welche von **zwei** Variablen abhängt, aber nur nach einer integriert werden soll. SciPy intergiert grundsätzlich immer nach der **ersten** Variable, die bei der Definition der Funktion genannt wird. Desweiteren müssen alle weiteren Variablen einen festen Zahlenwert erhalten, damit ein numerisches Ergebnis zustande kommen kann.

In [ ]:
import numpy as np
pi = np.pi

def g(x, y):
    return 1/(x**2 + y**2)

I = quad(g, 1, np.inf, args = (0)) # np.inf = "unendlich"

print(I)

Das obere Integral entspricht

$$ I = \int_1^{\infty} \mathrm{d}x \ g(x, 0) = \int_1^{\infty} \frac{\mathrm{d}x}{x^2} = -\left[ \frac{1}{x}\right]_1^{\infty} = 1 \ .$$

Über das keyword-Argument *args* gibt man die Werte aller restlichen Variablen in derselben Reihenfolge an, die sie in der Funktionsklammer haben.

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 1 </b>  </p> 
   
Berechnen Sie das folgende Integral mithilfe von SciPy: 
$$ I = \int_0^{2\pi} \mathrm{d}\theta \  h(\theta, 2) \ , \quad h(\theta, a) = \frac{1}{1 + a^2 - 2a\cos (\theta)}$$

In [ ]:
def h(theta, a):


I = quad()

print(I)

Es wird Sie vermutlich erstaunen, dass das obige Integral tatsächlich analytisch lösbar ist, sogar für ein allgemeines $a$. Man findet (ohne Beweis):

$$ \int_0^{2\pi} \frac{\mathrm{d}\theta}{1 + a^2 - 2a\cos (\theta)} = \frac{2\pi}{a^2 - 1} \ , \quad (|a| > 1)$$

Überprüfen Sie damit Ihr Ergebnis!

### II. Fourierreihen von Funktionen

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 2 </b>  </p> 
    
Definieren Sie nun die anfangs besprochene Funktion ```MyFourier(func, t, N)```, indem Sie den unteren Codeblock ergänzen. Die Funktion soll die folgenden Parameter entgegennehmen:
- **func**: Eine Pythonfunktion, die eine mathematische Funktion $f(t)$ beschreibt.
- **t**: Ein Zeitpunkt oder ein Array aus Zeitpunkten zu denen die Fourierreihe ausgewertet werden soll.
- **N**: Die Ordnung bis zu welcher die Fourierreihe berechnet werden soll.

Abschließend soll dann ein Array (gleiche Länge wie **t**) mit den Werten der Fourierreihe ausgegeben werden.
    
Zur Erinnerung:
    
$$ a_k = \frac{1}{\pi} \int_{0}^{2\pi} \mathrm{d}\tau \ f(\tau)\cos (k\tau) \ , \quad b_k = \frac{1}{\pi} \int_{0}^{2\pi} \mathrm{d}\tau \ f(\tau)\sin (k\tau) \ .$$

In [ ]:
def MyFourier(func, t, N):
    
    ### Definieren Sie die Integranden für die Integrale in a_k und b_k!
    def func_cos(tau, k):

    def func_sin(tau, k):

    
    ### Definieren Sie die Fourierkoeffizienten als Funktionen von k!
    def a(k):

    def b(k):

    
    ### Berechnen Sie nun die Fourierreihe!
    ### Der result-array is bereits auf den "nullten" (k = 0) Summanden initialisiert. 
    result = np.repeat(1/2 * 1/pi * quad(func, 0, 2 * pi)[0], len(t))
    for k in range():
        result += 
        
    return result

Mit der von Ihnen geschrieben Funktion haben wir die Möglichkeit uns mehrere Beispiele für Fourierreihen anzusehen und zu beobachten wie sich die Reihe für steigende $N$ der ursprünglichen Funktion immer weiter annähert. Im folgenden Codeblock befindet sich eine **vorgeschriebene** Funktion, die *MyFourier* implementiert um die Entwicklung der Fourierreihen zu visualisieren.

In [ ]:
import micropip
await micropip.install("ipywidgets")
from ipywidgets import interact
import matplotlib.pyplot as plt

def ShowFourier(func, t):
    @interact(N = (0, 20)) # konfiguriere das interaktive Widget 
    def show(N = 0):
        plt.figure(figsize = (8, 4)) # setze die Bildgröße fest
        plt.plot(t, func(t), color = "black", label = "func")
        plt.plot(t, MyFourier(func, t, N), color = "red", label = "Fourier")
        plt.xlabel("t"); plt.ylabel("y")
        plt.xlim(t[0], t[-1])
        plt.legend()
        plt.show()

Als erstes Beispiel betrachten wir die Funktion $f_1(t) = \sin (3t) + \cos(7t)$. Führen Sie den unteren Codeblock aus und verwenden Sie den Slider um sich die Fourierreihe für verschiedene N zu visualisieren.

In [ ]:
def f1(t):
    return np.sin(3 * t) + np.cos(7 * t)

dt = 0.1
t = np.arange(0, 10 * pi + dt, dt)
ShowFourier(f1, t)

Man kann klar beobachten, dass sich der Graph der Fourierreihe nur bei zwei bestimmten Werten von $N$ verändert: $3$ und $7$. Das bedeutet, dass alle Fourierkoeffizienten $a_k$, $b_k$ für $k \neq 3, 7$ verschwinden und dass, in unserem Falle, nur die Koeffizienten $a_7$ und $b_3$ ungleich null sind. Das ist nicht verwunderlich, denn unsere Eingangsfunktion $f_1(t)$ war schließlich eine Summe aus zwei trigonometrischen Funktionen mit Kreisfrequenzen $\omega_{\sin} = 3$ und $\omega_{\cos} = 7$. Jedoch kann jede beliebige $2\pi$-periodische (und auf $[0, 2\pi]$ integrierbare) Funktion durch eine Fourierreihe dargestellt werden, selbst eine die wenig mit trigonometrischen Funktionen zu tun hat.

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 3 </b>  </p> 
    
Nun sind Sie wieder dran. Definieren Sie die sogenannte Sägezahnfunktion $f_S$ in Python und verwenden Sie *ShowFourier* um sich die Fourierreihe zu visualisieren. Die Sägezahnfunktion wird im Intervall $[0, 2\pi)$ einfach durch $f_S = t$ beschrieben und wird periodisch nach vorne und hinten erweitert. 
> **Hinweis:** Bei der Definition der Funktion könnte der **modulo-Operator** (in Python ```%```) hilfreich sein. Er ist definiert als der Rest einer ganzzahligen Division, also $a \ \text{mod}  \ b = a - b \lfloor a / b \rfloor $. Ein Beispiel: $5 \ \text{mod} \ 2 = 1$.

In [ ]:
def fS(t):


ShowFourier(fS, t)

Es ist erkennbar, dass jeder Summand der Fourierreihe nun einen Beitrag leistet.

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 4 </b>  </p> 
    
Wiederholen Sie die vorherige Aufgabe mit der Rechteckfunktion $f_R(t)$. Im Intervall $[0, 2\pi)$ ist sie wie folgt definiert: 

$$ f_R(t) = \begin{cases} 1 \ , & t < \pi \\ 0  \ ,& t \geq \pi \end{cases} $$

In [ ]:
def fR(t):


ShowFourier(fR, t)

In den vorherigen beiden Aufgaben konnte man gut erkennen, wie die Fourierreihe sich der Ursprungsfunktion für steigende $N$ immer weiter annähert. Man sieht jedoch systematische Abweichungen. Bei der Rechteckfunktion erkennt man, dass die Fourierreihe die waagerechten Linien von $f_R(t)$ als Schwingungen um $y = 0$ bzw. $y = 1$ herum darstellt. Mit steigendem $N$ steigt die Frequenz dieser Schwingungen an, während die Amplitude sinkt. An den Kanten (Unstetigkeitsstellen von $f'_R(t)$) bleibt die Abweichung allerdings bestehen, egal wie hoch die Ordnung $N$ gewählt wird. Dieser Effekt trägt den Namen **Gibbs-Effekt**.

Der Gibbs-Effekt entsteht, wenn nicht alle Frequenzen die zu einer Funktion $f$ gehören in der Darstellung einer Fourierreihe berücksichtigt werden. Im ersten Beispiel bestand unsere gewählte Funktion gerade mal aus zwei überlagerten Frequenzen, wodurch für $N \geq 7$ keine Abweichungen mehr erkannt werden konnten. Die Sägezahn- und Rechtecksfunktion bestehen jedoch aus einer Überlagerung von **unendlich** vielen Frequenzen. Dadurch gibt es kein $N$ für das die Abweichungen verschwinden werden.

### III. Fourierreihen von Datensätzen

Wir möchten nun einen konkreten Anwendungsfall für Fourierreihen betrachten, wobei die **Fourierkoeffizienten** hier die tragende Rolle spielen werden. Dazu betrachten wir ein elektronisches Signal, welches sich aus sechs Schwingungen verschiedener Frequenzen zusammensetzt und stark von Untergrundrauschen gestört wird. Unser Ziel wird es sein die einzelnen Schwingungen aus dem Datensatz zu rekonstruieren.

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 5 </b>  </p> 
   
Machen Sie sich zunächst ein Bild von dem Datensatz. Der Datensatz deckt eine Zeitspanne von $200 \ \mathrm{ms}$ ab und enthält $200 \ 000$ Werte, die im gleichen Zeitabstand aufgenommen wurden. Plotten Sie nur die erste Millisekunde, damit die Datenpunkte nicht zu sehr gestaucht werden.

In [ ]:
### Lade den Datensatz
data = np.load("data.npy")

dt = 1e-6
t = np.arange(0, 1000, 1)

plt.figure(figsize = (14, 4)) # Definiert die Größe des Plots
plt.plot()



plt.show()

Um die verschiedenen Kreisfrequenzen zu ermitteln müssen wir die Fourierkoeffizienten bestimmen. Um uns das Leben zu erleichtern werden wir annehmen, dass sich alle Kreisfrequenzen im ganzzahligen kHz-Bereich bewegen mit $\omega_{\mathrm{max}} = 100 \ \mathrm{kHz}$. Somit müssen wir nur $100$ Koeffizientenpaare $a_k$, $b_k$ bestimmen. Da wir nun keine kontinuierliche Funktion mehr haben, sondern einen diskreten Datensatz müssen wir von Integralen zu Summen übergehen. Desweiteren werden wir über alle Datenpunkte summieren und müssen die Normierung dadurch entsprechend anpassen:

$$ a_\omega = \frac{1}{\pi} \int_{0}^{2\pi} \mathrm{d}\tau \ f(\tau)\cos (\omega \tau) \ \rightarrow \ 
\frac{2}{T} \int_{0}^{T} \mathrm{d}\tau \ f(\tau)\cos (\omega \tau) \ \rightarrow \ 
\frac{2}{N\Delta \tau} \sum_{i = 0}^{N} \Delta \tau f_i \cos (\omega \cdot i\Delta \tau)$$



<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 6 </b>  </p> 
   
Definieren Sie nun die Funktion ```MyFourierCoef(data, t, N)``` indem Sie die unten stehenden Codeblock ergänzen. Die Funktion funktioniert sehr ähnlich wie *MyFourier*, jedoch wird nun keine Funktion, sondern ein Datensatz übergeben. Desweiteren werden nur die Fourierkoeffizienten berechnet und ausgegeben. Plotten Sie anschließend die Fourierkoeffizienten in zwei getrennte Balkendiagramme. Dazu können Sie die Funktion ```plt.bar()``` verwenden.

In [ ]:
def MyFourierCoef(data, t, N):
    dt = t[1] - t[0]
    def a(k):
        
    def b(k):
        
    ### a_list und b_list enthalten alle für uns wichtigen Fourierkoeffizienten.
    a_list = []
    b_list = []
    
    ### Nun müssen alle Kreisfrequenzen abgearbeitet werden 
    for k in np.arange():
        a_list.append(a(k))
        b_list.append(b(k))
        
    return a_list, b_list

### Plotten Sie nun die Fourierkoeffizienten

Die Fourierkoeffizienten enthalten wichtige Informationen über die verschiedenen Schwingungen, wie (neben Frequenz) auch Amplitude und Phase. Um zu verstehen wie man diese Informationen extrahiert betrachten wir eine allgemeine Schwingung mit beliebiger Phase:

$$ U(t) = U_0 \sin (\omega t + \varphi) = U_0 \left( \sin (\omega t) \cos (\varphi) + \sin (\varphi) \cos (\omega t) \right) = \underbrace{U_0 \sin (\varphi)}_{=a_\omega} \cos (\omega t) + \underbrace{U_0 \cos (\varphi)}_{=b_\omega} \sin (\omega t) $$

Dadurch erhalten wir:

$$ U_0 = \sqrt{a_\omega^2 + b_\omega^2} \ , \quad \tan (\varphi) = \frac{a_\omega}{b_\omega}$$

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 7 </b>  </p> 
   
Berechnen Sie die Amplituden und Phasen aus den Fourierkoeffizienten und stellen Sie diese wieder in Balkendiagrammen dar. Überlegen Sie sich eine Methode um die sechs Frequenzen zu bestimmen und bestimmen Sie anschließend auch die zugehörigen Amplituden und Phasen.

Die folgende Tabelle enthält die tatsächlichen Parameter der verschiedenen Schwingungen. Vergleichen Sie Ihre Ergebnisse!

|                              | Signal 1    | Signal 2    | Signal 3    | Signal 4    | Signal 5    | Signal 6    |
| ---------------------------- | ----------- | ----------- | ----------- | ----------- | ----------- | ----------- |
| Kreisfrequenz $\omega$ [kHz] | 19          | 55          | 56          | 68          | 74          | 80          |
| Amplitude $U_0$ [V]          | 0.9         | 1.1         | 2.1         | 1.4         | 0.7         | 0.7         |
| Phase $\varphi$ [°]          | 19          | 0           | 66          | 0           | 71          | 90          |

<div style= "color: black;background-color: powderblue ;margin: 10 px auto; padding: 10px; border-radius: 10px">
    <p style="font-size:12pt; text-align:center; color:   black; background-color: lightskyblue ;margin: 10 px auto; padding: 10px; border-radius: 10px" id="1"><b>  Aufgabe 8 </b>  </p> 
   
Plotten Sie abschließend die erste Milisekunde des Datensatzes zusammen mit der Überlagerung der Schwingungen (Amplitude und Phase), die Sie herausgefiltert haben.